In [15]:
pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

query = """
SELECT "UserId", "Date", "ProcessName", "DurationSeconds"
FROM "DailySummaries"
ORDER BY "Date"
"""

raw_df = pd.read_sql(query, engine)
raw_df

,UserId,Date,ProcessName,DurationSeconds
0,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,Code.exe,372
1,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,chrome.exe,954
2,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,chrome.exe,6
3,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,SearchHost.exe,4
4,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,datagrip64.exe,46
5,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,explorer.exe,1
6,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,Code.exe,115
7,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,SnippingTool.exe,6
8,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,chrome.exe,161
9,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,Code.exe,121


In [2]:
import sys
sys.path.append('..')
from features import build_daily_dataframe, add_features,fill_missing_days, build_prediction_features

In [3]:
class SimpleApp:
    def __init__(self, process_name, duration_seconds):
        self.process_name = process_name
        self.duration_seconds = duration_seconds

class SimpleDay:
    def __init__(self, d, apps):
        self.date = d
        self.apps = apps

In [4]:
user_id = raw_df["UserId"].iloc[-1]
user_df = raw_df[raw_df["UserId"] == user_id]
user_df

,UserId,Date,ProcessName,DurationSeconds
2,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,chrome.exe,6
6,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,Code.exe,115
8,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,chrome.exe,161
9,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,Code.exe,121
10,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,datagrip64.exe,39
11,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,Code.exe,52
12,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,datagrip64.exe,36
13,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,chrome.exe,3
14,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,explorer.exe,4
15,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-20,explorer.exe,16


In [5]:
history = []
for d, group in user_df.groupby("Date"):
    apps = [SimpleApp(row.ProcessName, row.DurationSeconds) for row in group.itertuples()]
    history.append(SimpleDay(d, apps))

len(history)

5

In [6]:
daily_df = build_daily_dataframe(history)
daily_df

,date,total_seconds,top_app
0,2026-09-14,121,Code.exe
1,2026-09-15,321,chrome.exe
2,2026-09-16,95,Code.exe
3,2026-09-20,11476,chrome.exe
4,2026-09-25,12258,chrome.exe


In [7]:
filled_df = fill_missing_days(daily_df)
filled_df

,date,total_seconds,top_app
0,2026-09-14,121.0,Code.exe
1,2026-09-15,321.0,chrome.exe
2,2026-09-16,95.0,Code.exe
3,2026-09-17,0.0,none
4,2026-09-18,0.0,none
5,2026-09-19,0.0,none
6,2026-09-20,11476.0,chrome.exe
7,2026-09-21,0.0,none
8,2026-09-22,0.0,none
9,2026-09-23,0.0,none


In [8]:
feature_df = add_features(filled_df)
feature_df

,date,total_seconds,top_app,is_weekend,previous_day_total,rolling_7day_avg,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6
0,2026-09-14,121.0,Code.exe,0,NaN,NaN,False,False,False,False,False,False
1,2026-09-15,321.0,chrome.exe,0,121.0,121.000000,True,False,False,False,False,False
2,2026-09-16,95.0,Code.exe,0,321.0,221.000000,False,True,False,False,False,False
3,2026-09-17,0.0,none,0,95.0,179.000000,False,False,True,False,False,False
4,2026-09-18,0.0,none,0,0.0,134.250000,False,False,False,True,False,False
5,2026-09-19,0.0,none,1,0.0,107.400000,False,False,False,False,True,False
6,2026-09-20,11476.0,chrome.exe,1,0.0,89.500000,False,False,False,False,False,True
7,2026-09-21,0.0,none,0,11476.0,1716.142857,False,False,False,False,False,False
8,2026-09-22,0.0,none,0,0.0,1698.857143,True,False,False,False,False,False
9,2026-09-23,0.0,none,0,0.0,1653.000000,False,True,False,False,False,False


In [9]:
feature_df = add_features(filled_df)

In [10]:
feature_df

,date,total_seconds,top_app,is_weekend,previous_day_total,rolling_7day_avg,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6
0,2026-09-14,121.0,Code.exe,0,NaN,NaN,False,False,False,False,False,False
1,2026-09-15,321.0,chrome.exe,0,121.0,121.000000,True,False,False,False,False,False
2,2026-09-16,95.0,Code.exe,0,321.0,221.000000,False,True,False,False,False,False
3,2026-09-17,0.0,none,0,95.0,179.000000,False,False,True,False,False,False
4,2026-09-18,0.0,none,0,0.0,134.250000,False,False,False,True,False,False
5,2026-09-19,0.0,none,1,0.0,107.400000,False,False,False,False,True,False
6,2026-09-20,11476.0,chrome.exe,1,0.0,89.500000,False,False,False,False,False,True
7,2026-09-21,0.0,none,0,11476.0,1716.142857,False,False,False,False,False,False
8,2026-09-22,0.0,none,0,0.0,1698.857143,True,False,False,False,False,False
9,2026-09-23,0.0,none,0,0.0,1653.000000,False,True,False,False,False,False


In [11]:
from features import build_training_data

X, y_total, y_top_app = build_training_data(feature_df)
print(X.shape)
print(y_total.shape, y_top_app.shape)

(11, 8)
(11,) (11,)


In [12]:
model_df = feature_df.dropna()

feature_columns = ["previous_day_total", "rolling_7day_avg",
                    "dow_1", "dow_2", "dow_3", "dow_4", "dow_5", "dow_6"]

X = model_df[feature_columns]
y = model_df["total_seconds"]

X

,previous_day_total,rolling_7day_avg,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6
1,121.0,121.000000,True,False,False,False,False,False
2,321.0,221.000000,False,True,False,False,False,False
3,95.0,179.000000,False,False,True,False,False,False
4,0.0,134.250000,False,False,False,True,False,False
5,0.0,107.400000,False,False,False,False,True,False
6,0.0,89.500000,False,False,False,False,False,True
7,11476.0,1716.142857,False,False,False,False,False,False
8,0.0,1698.857143,True,False,False,False,False,False
9,0.0,1653.000000,False,True,False,False,False,False
10,0.0,1639.428571,False,False,True,False,False,False


In [13]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[ -0.44, 1.94,-3343.64,..., 2643.1 ,-1982.84, 9527.81]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['previous_day_total','rolling_7day_avg','dow_1',...,'dow_4','dow_5', 'dow_6']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1763
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(7)


In [14]:
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

Coefficients: [-4.36848128e-01  1.93621515e+00 -3.34363935e+03 -3.46531713e+03
 -3.50844219e+03  2.64310448e+03 -1.98284286e+03  9.52781372e+03]
Intercept: 1762.747009841568


In [15]:
pred_features, tomorrow_date = build_prediction_features(filled_df)
print(tomorrow_date)
pred_features

2026-09-26 00:00:00


,previous_day_total,rolling_7day_avg,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6
0,12258.0,3390.571429,0,0,0,0,1,0


In [16]:
prediction = model.predict(pred_features)
print("Predicted total seconds for", tomorrow_date, ":", prediction[0])

Predicted total seconds for 2026-09-26 00:00:00 : 989.8955523053526


In [17]:
y_class = model_df["top_app"]
y_class

1     chrome.exe
2       Code.exe
3           none
4           none
5           none
6     chrome.exe
7           none
8           none
9           none
10          none
11    chrome.exe
Name: top_app, dtype: str

In [18]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X, y_class)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [19]:
top_app_prediction = clf.predict(pred_features)
print("Predicted top app for", tomorrow_date, ":", top_app_prediction[0])

Predicted top app for 2026-09-26 00:00:00 : none


In [20]:
import pandas as pd

probs = clf.predict_proba(pred_features)[0]
pd.Series(probs, index=clf.classes_).sort_values(ascending=False)

none          9.998408e-01
Code.exe      1.592093e-04
chrome.exe    3.511633e-14
dtype: float64

In [ ]:
from train_model import train_models, save_models, load_models

regressor, classifier = train_models(feature_df)
save_models(user_id, regressor, classifier)

loaded_regressor, loaded_classifier = load_models(user_id)

print(loaded_regressor.predict(pred_features)[0])
print(loaded_classifier.predict(pred_features)[0])

ImportError: cannot import name 'train_models' from 'train_model' (c:\Users\Lenovo\Desktop\Fakultet-Ja\Ja\MoiLichniProekti\ScreenTimeTracker\ml-service\notebooks\..\train_model.py)